# 🏠 Análisis de Arriendos en Santiago, Chile — Mayo 2025

**Autor:** Emilio Rubina Salinas  
**LinkedIn:** [linkedin.com/in/emilio-rubina-salinas-b1436a253](https://www.linkedin.com/in/emilio-rubina-salinas-b1436a253/)  
**Fuente:** Yapo.cl — Mayo 2025  
**Dataset:** 12.690 publicaciones de departamentos en la Región Metropolitana

## Objetivo
Análisis exploratorio del mercado de arriendos en Santiago para identificar:
- Precio promedio por comuna
- Relación precio vs metros cuadrados
- Comunas más económicas y más caras
- Distribución de tamaños y dormitorios
- Precio por m² como métrica comparativa real

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BLUE   = '#1A56A0'
GREEN  = '#3B6D11'
AMBER  = '#BA7517'
RED    = '#A32D2D'
GRAY   = '#888780'

print('Librerías cargadas ✓')

Librerías cargadas ✓


## 1. Carga y exploración inicial

In [5]:
df = pd.read_csv('data/apartments_santiago_clean.csv')

print(f'Total registros: {len(df)}')
print(f'Con precio válido: {df["Precio_CLP"].notna().sum()}')
print(f'Con área válida: {df["Area_m2"].notna().sum()}')
print(f'Con precio/m2: {df["Precio_por_m2"].notna().sum()}')
print(f'Comunas únicas: {df["Comuna"].nunique()}')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/apartments_santiago_clean.csv'

In [3]:
# Estadísticas generales
df_precio = df[df['Precio_CLP'].notna()]
print('=== ESTADÍSTICAS DE PRECIO (CLP) ===')
print(f'Mínimo:   ${df_precio["Precio_CLP"].min():,.0f}')
print(f'Promedio: ${df_precio["Precio_CLP"].mean():,.0f}')
print(f'Mediana:  ${df_precio["Precio_CLP"].median():,.0f}')
print(f'Máximo:   ${df_precio["Precio_CLP"].max():,.0f}')

NameError: name 'df' is not defined

## 2. Distribución de precios

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribución de Precios de Arriendo — Santiago 2025', fontsize=14, fontweight='bold', color=BLUE)

df_hist = df_precio[df_precio['Precio_CLP'] <= 1000000]

axes[0].hist(df_hist['Precio_CLP'], bins=40, color=BLUE, edgecolor='white', linewidth=0.5)
axes[0].axvline(df_hist['Precio_CLP'].median(), color=RED, linestyle='--', linewidth=2, label=f'Mediana: ${df_hist["Precio_CLP"].median():,.0f}')
axes[0].axvline(df_hist['Precio_CLP'].mean(), color=AMBER, linestyle='--', linewidth=2, label=f'Promedio: ${df_hist["Precio_CLP"].mean():,.0f}')
axes[0].set_xlabel('Precio CLP', fontsize=11)
axes[0].set_ylabel('Número de publicaciones', fontsize=11)
axes[0].set_title('Histograma de precios (hasta $1M)', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

df_precio.boxplot(column='Precio_CLP', ax=axes[1], 
    boxprops=dict(color=BLUE), medianprops=dict(color=RED, linewidth=2),
    whiskerprops=dict(color=GRAY), capprops=dict(color=GRAY),
    flierprops=dict(marker='o', markerfacecolor=AMBER, markersize=3, alpha=0.3))
axes[1].set_title('Boxplot de precios', fontweight='bold')
axes[1].set_ylabel('Precio CLP')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('01_distribucion_precios.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 La mediana ($300K) es más representativa que el promedio ($454K) por los outliers de lujo.')

NameError: name 'plt' is not defined

## 3. Precio promedio por comuna (Top 15)

In [5]:
# Solo comunas con al menos 30 publicaciones con precio
comuna_stats = df_precio.groupby('Comuna').agg(
    Precio_Promedio=('Precio_CLP', 'median'),
    Total_Publicaciones=('Precio_CLP', 'count'),
    Precio_m2_Mediana=('Precio_por_m2', 'median')
).reset_index()

comunas_validas = comuna_stats[comuna_stats['Total_Publicaciones'] >= 30].sort_values('Precio_Promedio', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))

colors = []
for _, row in comunas_validas.iterrows():
    if row['Precio_Promedio'] >= 500000: colors.append(RED)
    elif row['Precio_Promedio'] >= 350000: colors.append(AMBER)
    elif row['Precio_Promedio'] >= 250000: colors.append(BLUE)
    else: colors.append(GREEN)

bars = ax.barh(comunas_validas['Comuna'], comunas_validas['Precio_Promedio'], 
               color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, comunas_validas['Precio_Promedio']):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f'${val/1000:.0f}K', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Precio Mediano CLP', fontsize=11)
ax.set_title('🏘️ Precio mediano de arriendo por comuna — Santiago 2025',
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax.invert_yaxis()
ax.set_xlim(0, comunas_validas['Precio_Promedio'].max() * 1.18)

patch_caro = mpatches.Patch(color=RED, label='Premium (>$500K)')
patch_medio_alto = mpatches.Patch(color=AMBER, label='Medio-alto ($350K-$500K)')
patch_medio = mpatches.Patch(color=BLUE, label='Medio ($250K-$350K)')
patch_eco = mpatches.Patch(color=GREEN, label='Económico (<$250K)')
ax.legend(handles=[patch_caro, patch_medio_alto, patch_medio, patch_eco], 
          loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('02_precio_por_comuna.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'df_precio' is not defined

## 4. Precio por m² — El indicador real de valor

In [6]:
df_m2 = df[df['Precio_por_m2'].notna() & df['Precio_por_m2'].between(1000, 30000)]

comunas_m2 = df_m2.groupby('Comuna').agg(
    Precio_m2_Mediana=('Precio_por_m2', 'median'),
    Total=('Precio_por_m2', 'count')
).reset_index()
comunas_m2 = comunas_m2[comunas_m2['Total'] >= 20].sort_values('Precio_m2_Mediana', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(11, 7))

cmap = plt.cm.RdYlGn_r
norm = plt.Normalize(comunas_m2['Precio_m2_Mediana'].min(), comunas_m2['Precio_m2_Mediana'].max())
colors_m2 = [cmap(norm(v)) for v in comunas_m2['Precio_m2_Mediana']]

bars = ax.barh(comunas_m2['Comuna'][::-1], comunas_m2['Precio_m2_Mediana'][::-1],
               color=colors_m2[::-1], edgecolor='white')

for bar, val in zip(bars, comunas_m2['Precio_m2_Mediana'][::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}/m²', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Precio mediano por m² (CLP)', fontsize=11)
ax.set_title('💰 Precio por m² por comuna — Indicador real de valor',
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
ax.set_xlim(0, comunas_m2['Precio_m2_Mediana'].max() * 1.22)

plt.tight_layout()
plt.savefig('03_precio_por_m2.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Precio/m² es más justo que precio total — permite comparar deptos de distintos tamaños.')

NameError: name 'df' is not defined

## 5. Relación Precio vs Área m²

In [7]:
df_scatter = df[
    df['Precio_CLP'].notna() & 
    df['Area_m2'].notna() &
    df['Precio_CLP'].between(100000, 1500000) &
    df['Area_m2'].between(20, 150)
].copy()

fig, ax = plt.subplots(figsize=(11, 7))

comunas_top = df_scatter['Comuna'].value_counts().head(6).index
colors_scatter = [BLUE, RED, GREEN, AMBER, '#9B59B6', '#E67E22']

for i, comuna in enumerate(comunas_top):
    mask = df_scatter['Comuna'] == comuna
    ax.scatter(df_scatter[mask]['Area_m2'], df_scatter[mask]['Precio_CLP'],
               alpha=0.4, s=20, color=colors_scatter[i], label=comuna)

mask_others = ~df_scatter['Comuna'].isin(comunas_top)
ax.scatter(df_scatter[mask_others]['Area_m2'], df_scatter[mask_others]['Precio_CLP'],
           alpha=0.15, s=10, color=GRAY, label='Otras comunas')

# Línea de tendencia
z = np.polyfit(df_scatter['Area_m2'], df_scatter['Precio_CLP'], 1)
p = np.poly1d(z)
x_line = np.linspace(20, 150, 100)
ax.plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--', 
        label=f'Tendencia: +${z[0]:,.0f}/m²')

ax.set_xlabel('Área (m²)', fontsize=11)
ax.set_ylabel('Precio CLP', fontsize=11)
ax.set_title('📐 Relación Precio vs Área — Por comuna',
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('04_precio_vs_area.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💡 Por cada m² adicional, el precio sube aproximadamente ${z[0]:,.0f} CLP en promedio.')

NameError: name 'df' is not defined

## 6. Distribución por dormitorios y tamaño

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Composición del Mercado de Arriendos', fontsize=14, fontweight='bold', color=BLUE)

# Dormitorios
dorm_counts = df[df['Dormitorios'].between(1,4)]['Dormitorios'].value_counts().sort_index()
dorm_labels = ['1 dormitorio', '2 dormitorios', '3 dormitorios', '4 dormitorios']
dorm_colors = [BLUE, GREEN, AMBER, RED]
bars1 = axes[0].bar(dorm_labels, dorm_counts.values, color=dorm_colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars1, dorm_counts.values):
    pct = round(val / dorm_counts.sum() * 100)
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{val:,}\n({pct}%)', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Por número de dormitorios', fontweight='bold')
axes[0].set_ylabel('Publicaciones')
axes[0].tick_params(axis='x', rotation=10)

# Tamaño
tam_order = ['Studio / Mini (<35m2)', 'Pequeño (35-54m2)', 'Mediano (55-74m2)', 'Grande (75-99m2)', 'Premium (100m2+']
tam_counts = df['Tamaño'].value_counts().reindex([t for t in tam_order if t in df['Tamaño'].values], fill_value=0)
tam_colors2 = [GREEN, BLUE, AMBER, RED, '#6C3483']
wedges, texts, autotexts = axes[1].pie(
    tam_counts.values, labels=[t.split('(')[0].strip() for t in tam_counts.index],
    autopct='%1.1f%%', colors=tam_colors2[:len(tam_counts)],
    pctdistance=0.75, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for t in autotexts:
    t.set_fontsize(9)
    t.set_fontweight('bold')
axes[1].set_title('Por tamaño del departamento', fontweight='bold')

plt.tight_layout()
plt.savefig('05_dormitorios_tamano.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

## 7. Precio por dormitorio — ¿Cuánto cuesta cada dormitorio adicional?

In [9]:
df_dorm = df[df['Precio_CLP'].notna() & df['Dormitorios'].between(1, 4)]

dorm_precio = df_dorm.groupby('Dormitorios')['Precio_CLP'].agg(['median','mean','count']).reset_index()
dorm_precio.columns = ['Dormitorios', 'Mediana', 'Promedio', 'Total']

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(dorm_precio))
width = 0.35
bars_med = ax.bar(x - width/2, dorm_precio['Mediana'], width, label='Mediana', color=BLUE, edgecolor='white')
bars_avg = ax.bar(x + width/2, dorm_precio['Promedio'], width, label='Promedio', color=AMBER, edgecolor='white')

for bar in bars_med:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000,
            f'${bar.get_height()/1000:.0f}K', ha='center', fontsize=9, fontweight='bold', color=BLUE)
for bar in bars_avg:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000,
            f'${bar.get_height()/1000:.0f}K', ha='center', fontsize=9, fontweight='bold', color=AMBER)

ax.set_xticks(x)
ax.set_xticklabels([f'{int(d)} dorm.' for d in dorm_precio['Dormitorios']])
ax.set_ylabel('Precio CLP')
ax.set_title('💵 Precio de arriendo por número de dormitorios',
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('06_precio_por_dormitorios.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'df' is not defined

## 8. Conclusiones

In [10]:
df_precio = df[df['Precio_CLP'].notna()]
df_m2 = df[df['Precio_por_m2'].notna() & df['Precio_por_m2'].between(1000, 30000)]

print('='*60)
print('CONCLUSIONES DEL ANÁLISIS')
print('='*60)

conclusiones = [
    f'1. Precio mediano de arriendo en Santiago: ${df_precio["Precio_CLP"].median():,.0f} CLP',
    f'2. Precio promedio (inflado por lujo): ${df_precio["Precio_CLP"].mean():,.0f} CLP',
    f'3. Precio mediano por m²: ${df_m2["Precio_por_m2"].median():,.0f} CLP',
    f'4. Comuna más cara: {df.groupby("Comuna")["Precio_CLP"].median().idxmax()}',
    f'5. El 50% de los deptos tiene 1-2 dormitorios',
    f'6. Santiago centro concentra el {round(df[df["Comuna"]=="Santiago"].shape[0]/len(df)*100)}% de las publicaciones',
    f'7. Solo el {round(df["Precio_CLP"].notna().sum()/len(df)*100)}% de las publicaciones tiene precio explícito',
    f'8. Datos scrapeados de Yapo.cl — pueden incluir publicaciones duplicadas o desactualizadas',
]

for c in conclusiones:
    print(f'  {c}')

NameError: name 'df' is not defined